In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Configurar la ruta absoluta al archivo Excel
# (Al estar en la carpeta 'notebooks', subimos un nivel para llegar a la raíz)
BASE_DIR = os.path.dirname(os.getcwd())
path_data = os.path.join(BASE_DIR, "Base_de_datos.xlsx")

# 2. Cargar el dataset
df = pd.read_excel(path_data)
print(f"Dataset cargado para el EDA. Dimensiones: {df.shape}")

Dataset cargado para el EDA. Dimensiones: (10763, 23)


In [6]:
# =====================================================================
# 1. EXPLORACIÓN INICIAL, UNIFICACIÓN DE NULOS Y CONVERSIÓN DE TIPOS
# =====================================================================

# --- PASO 1: Unificación de formatos de Nulos ocultos ---
valores_nulos_ocultos = ["?", "unknown", "None", "null", "NA", " ", ""]
df.replace(valores_nulos_ocultos, np.nan, inplace=True)

# --- PASO 2: Eliminación de variables irrelevantes ---
# No tienes IDs a la vista, pero si existiera alguna columna de índice oculta la removemos.
columnas_irrelevantes = ['id', 'index'] 
df.drop(columns=columnas_irrelevantes, inplace=True, errors='ignore')

# --- PASO 3: Corrección y Conversión de Tipos de Datos Uniformes ---
# Convertimos las variables categóricas de tipo 'object' a formato 'category'
columnas_categoricas = ['tipo_credito', 'tipo_laboral', 'tendencia_ingresos']
for col in columnas_categoricas:
    if col in df.columns:
        df[col] = df[col].astype('category')

# Convertimos la variable objetivo a booleana/dicotómica (0 = False, 1 = True)
if 'Pago_atiempo' in df.columns:
    df['Pago_atiempo'] = df['Pago_atiempo'].astype(bool)

# --- PASO 4: Reporte Técnico de Validación ---
print("=== ESTRUCTURA DEPURADA Y CARACTERIZACIÓN FINANCIERA ===")
resumen_tipos = pd.DataFrame({
    'Tipo Técnico': df.dtypes,
    'Cantidad Nulos Reales': df.isnull().sum(),
    'Valores Únicos': df.nunique()
})
display(resumen_tipos)

=== ESTRUCTURA DEPURADA Y CARACTERIZACIÓN FINANCIERA ===


,Tipo Técnico,Cantidad Nulos Reales,Valores Únicos
tipo_credito,category,0,6
fecha_prestamo,datetime64[ns],0,10758
capital_prestado,float64,0,7306
plazo_meses,int64,0,18
edad_cliente,int64,0,54
tipo_laboral,category,0,2
salario_cliente,int64,0,1385
total_otros_prestamos,int64,0,1538
cuota_pactada,int64,0,9836
puntaje,float64,0,248


### 📊 Caracterización y Clasificación Teórica de las Variables

Tras realizar la revisión inicial en la tabla interactiva y aplicar el procesamiento de datos, clasificamos formalmente las 23 columnas según su naturaleza estadística:

1. **Variables Numéricas:**
   * **Continuas:** `capital_prestado`, `salario_cliente`, `puntaje`, `puntaje_datacredito`, `saldo_mora`, `saldo_total`, `saldo_principal`, `saldo_mora_codeudor`, y `promedio_ingresos_datacredito`. Representan importes monetarios o scores de riesgo continuos.
   * **Discretas:** `plazo_months`, `edad_client`, `total_otros_prestamos`, `cuota_pactada`, `cant_creditosvigentes`, `huella_consulta`, `creditos_sectorFinanciero`, `creditos_sectorCooperativo`, y `creditos_sectorReal`. Representan conteos enteros, meses o cantidades de productos.

2. **Variables Categóricas:**
   * **Nominales (Politómicas):** `tipo_credito` (6 tipos) y `tipo_laboral` (2 tipos). Definen atributos cualitativos sin un orden jerárquico inherente.
   * **Ordinales:** `tendencia_ingresos` (46 valores únicos). Representa categorías que describen un comportamiento de comportamiento o escala temporal.

3. **Variables Temporales:**
   * `fecha_prestamo`: Registrada de manera uniforme bajo el tipo `datetime64[ns]`.

4. **Variables Dicotómicas / Binarias (Target):**
   * `Pago_atiempo`: Posee únicamente 2 valores únicos. Representa nuestra variable objetivo (si el cliente pagó a tiempo o cayó en mora).

### 🛠️ Acciones de Limpieza y Diagnóstico Aplicadas:
* **Estandarización de Tipos:** Se tipificó `Pago_atiempo` como booleano y las columnas de texto como categorías estructuradas para optimizar el uso de memoria.
* **Diagnóstico de Nulos:** Se detectó que las variables `promedio_ingresos_datacredito` y `tendencia_ingresos` concentran la mayor cantidad de datos faltantes (~2,930 registros). Estas requerirán estrategias de imputación o tratamiento específico en la siguiente fase para no perder el 27% del dataset.